In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import t as t_dist

data = pd.read_csv('data_in_class_three.csv')
y = data['ETS_scaled_score']
x = data['ECON_GPA']

model = sm.OLS(y, sm.add_constant(x)).fit()

alpha_hat, beta_hat = model.params
S = np.sqrt(model.mse_resid)
n = int(model.nobs)
x_bar = x.mean()
S_xx = np.sum((x - x_bar)**2)

print(f"Estimated model:  Ŷ = {alpha_hat:.2f} + {beta_hat:.2f}·x")
print(f"n = {n},  x̄ = {x_bar:.3f},  S_xx = {S_xx:.2f},  S = {S:.3f}")

Estimated model:  Ŷ = 118.81 + 12.93·x
n = 258,  x̄ = 2.920,  S_xx = 88.52,  S = 12.138


# ETS In-Class Exercise — Answer Key

The Economics ETS exam is given to all graduating Economics seniors at Cal Poly. We model the relationship between a student's **ETS score** ($Y$) and their **Econ GPA** ($x$):

$$Y \mid x \;\sim\; N(\alpha + \beta\, x,\;\sigma^2)$$

We have data on 258 students. We estimate $\hat{\alpha}, \hat{\beta}$, and $S$ by OLS, then answer the questions for a student with Econ GPA $x_0 = 3.4$.

## (a) Estimate the ETS score of a student with Econ GPA of 3.4

This is the point estimate $\hat{\alpha} + \hat{\beta}\,x_0$.

In [2]:
x_0 = 3.4
y_hat = alpha_hat + beta_hat * x_0

print(f"Estimated ETS score at x₀ = {x_0}:")
print(f"  α̂ + β̂·x₀ = {alpha_hat:.2f} + {beta_hat:.2f}·{x_0} = {y_hat:.2f}")

Estimated ETS score at x₀ = 3.4:
  α̂ + β̂·x₀ = 118.81 + 12.93·3.4 = 162.75


## (b) 95% prediction interval for the ETS score of a student with Econ GPA of 3.4

This is a **prediction interval** for an individual observation $Y_0$ (eq. 12.2.36):

$$\hat{\alpha} + \hat{\beta}\,x_0 \;\pm\; t_{n-2,\,0.025}\; S\,\sqrt{1 + \frac{1}{n} + \frac{(x_0 - \bar{x})^2}{S_{xx}}}$$

In [3]:
df = n - 2
t_crit = t_dist.ppf(0.975, df)

h = 1/n + (x_0 - x_bar)**2 / S_xx
pi_half = t_crit * S * np.sqrt(1 + h)

print(f"95% Prediction Interval for Y₀ (individual student):")
print(f"  t_{{0.025, {df}}} = {t_crit:.4f}")
print(f"  1/n + (x₀ - x̄)²/S_xx = {h:.5f}")
print(f"  S·√(1 + h) = {S * np.sqrt(1 + h):.3f}")
print(f"  [{y_hat - pi_half:.2f},  {y_hat + pi_half:.2f}]")

95% Prediction Interval for Y₀ (individual student):
  t_{0.025, 256} = 1.9693
  1/n + (x₀ - x̄)²/S_xx = 0.00648
  S·√(1 + h) = 12.178
  [138.77,  186.73]


## (c) Interpret this interval

With 95% probability, the ETS score of a student with Econ GPA of 3.4 falls in this interval. It is wide because it accounts for *both* the uncertainty in estimating the regression line *and* the inherent variability ($\sigma^2$) of individual students around the line.

## (b') 95% confidence interval for the *average* ETS score of all students with Econ GPA of 3.4

This is a **confidence interval** for the conditional mean $\alpha + \beta x_0$ (eq. 12.2.35):

$$\hat{\alpha} + \hat{\beta}\,x_0 \;\pm\; t_{n-2,\,0.025}\; S\,\sqrt{\frac{1}{n} + \frac{(x_0 - \bar{x})^2}{S_{xx}}}$$

In [4]:
ci_half = t_crit * S * np.sqrt(h)

print(f"95% Confidence Interval for E[Y|x₀] (average student):")
print(f"  S·√(h) = {S * np.sqrt(h):.3f}")
print(f"  [{y_hat - ci_half:.2f},  {y_hat + ci_half:.2f}]")
print(f"\nCompare widths:")
print(f"  CI width: {2*ci_half:.2f}")
print(f"  PI width: {2*pi_half:.2f}")

95% Confidence Interval for E[Y|x₀] (average student):
  S·√(h) = 0.977
  [160.83,  164.68]

Compare widths:
  CI width: 3.85
  PI width: 47.96


## (d) Interpret this interval

With 95% confidence, the *average* ETS score of all students with Econ GPA of 3.4 falls in this interval. It is much narrower than the prediction interval because it only captures uncertainty about the location of the regression line — not the scatter of individual students around it.

## (e) & (f) Probability of meeting and exceeding expectations

A student **meets expectations** if $Y \geq 157$ and **exceeds expectations** if $Y \geq 169$.

Under the model, $Y_0 \mid x_0 \sim N(\hat{\alpha} + \hat{\beta}\,x_0,\; S^2(1 + h))$, so:

$$P(Y_0 \geq c) = 1 - \Phi\!\left(\frac{c - (\hat{\alpha} + \hat{\beta}\,x_0)}{S\sqrt{1 + h}}\right)$$

where $\Phi$ is the standard normal CDF (or strictly, the $t_{n-2}$ CDF — but with 256 degrees of freedom the difference is negligible).

In [5]:
se_pred = S * np.sqrt(1 + h)

for cutoff, label in [(157, "meets expectations"), (169, "exceeds expectations")]:
    t_score = (cutoff - y_hat) / se_pred
    prob = 1 - t_dist.cdf(t_score, df)
    print(f"(e/f) P(Y₀ ≥ {cutoff} | x₀ = {x_0})  [{label}]")
    print(f"      t-score = ({cutoff} - {y_hat:.2f}) / {se_pred:.3f} = {t_score:.4f}")
    print(f"      P = {prob:.4f}  ({prob*100:.1f}%)\n")

(e/f) P(Y₀ ≥ 157 | x₀ = 3.4)  [meets expectations]
      t-score = (157 - 162.75) / 12.178 = -0.4725
      P = 0.6815  (68.1%)

(e/f) P(Y₀ ≥ 169 | x₀ = 3.4)  [exceeds expectations]
      t-score = (169 - 162.75) / 12.178 = 0.5130
      P = 0.3042  (30.4%)

